In [ ]:
!pip install scikit-surprise pandas numpy

In [ ]:
import pandas as pd
import numpy as np

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy
from collections import defaultdict

In [ ]:
# Load built-in dataset
data = Dataset.load_builtin('ml-100k')

# Split dataset
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

Dataset ml-100k could not be found. Do you want to download it? [Y/n] Y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k


In [ ]:
model = SVD()

model.fit(trainset)

In [ ]:
predictions = model.test(testset)

rmse = accuracy.rmse(predictions)
print("RMSE:", rmse)

RMSE: 0.9363
RMSE: 0.9362804957043518


In [ ]:
def precision_recall_at_k(predictions, k=5, threshold=3.5):
    user_est_true = defaultdict(list)

    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = {}
    recalls = {}

    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        top_k = user_ratings[:k]

        relevant = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        recommended = sum((est >= threshold) for (est, _) in top_k)

        relevant_and_recommended = sum(
            ((true_r >= threshold) and (est >= threshold))
            for (est, true_r) in top_k
        )

        precisions[uid] = relevant_and_recommended / recommended if recommended != 0 else 0
        recalls[uid] = relevant_and_recommended / relevant if relevant != 0 else 0

    return np.mean(list(precisions.values())), np.mean(list(recalls.values()))

precision, recall = precision_recall_at_k(predictions)

print("Precision@5:", precision)
print("Recall@5:", recall)

Precision@5: 0.7420744680851064
Recall@5: 0.42262808908362315


In [ ]:
# Download movie metadata
!wget https://files.grouplens.org/datasets/movielens/ml-100k/u.item

# Load movie names
movie_df = pd.read_csv(
    'u.item',
    sep='|',
    encoding='latin-1',
    header=None,
    usecols=[0, 1],
    names=['movie_id', 'title']
)

movie_df.head()

--2026-03-29 19:32:58--  https://files.grouplens.org/datasets/movielens/ml-100k/u.item
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 236344 (231K)
Saving to: ‘u.item’

u.item              100%[===================>] 230.80K  --.-KB/s    in 0.09s   

2026-03-29 19:32:58 (2.38 MB/s) - ‘u.item’ saved [236344/236344]



,movie_id,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


In [ ]:
def recommend_movies(user_id, n=10):
    all_movie_ids = movie_df['movie_id'].unique()

    predictions = []

    for movie_id in all_movie_ids:
        pred = model.predict(user_id, movie_id)
        predictions.append((movie_id, pred.est))

    # Sort by rating
    predictions.sort(key=lambda x: x[1], reverse=True)

    top_n = predictions[:n]

    # Show results
    print(f"\nTop {n} recommendations for User {user_id}:\n")

    for movie_id, rating in top_n:
        title = movie_df[movie_df['movie_id'] == movie_id]['title'].values
        if len(title) > 0:
            print(f"{title[0]} → Predicted Rating: {round(rating,2)}")

In [ ]:
recommend_movies(user_id=196, n=10)


Top 10 recommendations for User 196:

Toy Story (1995) → Predicted Rating: 3.53
GoldenEye (1995) → Predicted Rating: 3.53
Four Rooms (1995) → Predicted Rating: 3.53
Get Shorty (1995) → Predicted Rating: 3.53
Copycat (1995) → Predicted Rating: 3.53
Shanghai Triad (Yao a yao yao dao waipo qiao) (1995) → Predicted Rating: 3.53
Twelve Monkeys (1995) → Predicted Rating: 3.53
Babe (1995) → Predicted Rating: 3.53
Dead Man Walking (1995) → Predicted Rating: 3.53
Richard III (1995) → Predicted Rating: 3.53
